In [29]:
import numpy as np

* You will only be given these four data structures
* No other template code or coding-by-contract will be provided
* It may benefit you to utlize Object-Oriented Programming (OOP) as you will be developing out the complete HMM suite throughout the HMM modules
* Make no assumptions as to the number of hidden states you will be given
* Make no assumptions as to the number of distinct observations you will be given
* Make no assumptions that the data structures will be modeling CpG islands (these were just examples)

In [83]:
class HiddenMarkovModel:
    def __init__(self, initial_probs, transition_probs, emission_probs):
        self.states = []
        self.states = list(initial_probs.keys())
        self.initial_probs = initial_probs
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs
    def get_states(self):
        return self.states

    def get_initial_probs(self, state):
        return self.initial_probs[state]

    def get_transition_probs(self, state):
        return self.transition_probs[state]

    def get_emission_probs(self, state):
        return self.emission_probs[state]


In [284]:
def viterbi_algorithm(observations, initial_probs, transition_probs, emission_probs):
    if type(observations) != list:
        observations = [observations]

    # Create HMM class
    hmm = HiddenMarkovModel(initial_probs, transition_probs, emission_probs)
    print(hmm.get_states())
    optimal_path = []
    for observation in observations:
        viterbi_matrix, traceback_matrix = build_viterbi_matrix(observation, hmm)
        print(f"Observation: {observation}")
        print(f"Viterbi matrix:\n{viterbi_matrix}")
        print(f"Traceback matrix:\n{traceback_matrix}")
        optimal_path_index = viterbi_traceback(viterbi_matrix, traceback_matrix)
        print(f"Optimal path index:\n{optimal_path_index}")
        path = convert_index_to_states(optimal_path_index, hmm.get_states())
        print(f"Optimal path:\n{path}\n")
        optimal_path.append(path)


    return optimal_path



In [285]:
def build_viterbi_matrix(observations, hmm):
    ####### Intiialization ########

    # Get states and observations from class object
    states = hmm.get_states()

    #Initialize viterbi and traceback matrix
    prob_matrix = np.zeros((len(states), len(observations)), dtype = float)
    traceback_matrix = np.zeros((len(states), len(observations)), dtype = int)

    ####### Iteration ########
    for i, observation in enumerate(observations):
        for j, state in enumerate(states):
            state_init_probs = hmm.get_initial_probs(state)
            state_emit_probs = hmm.get_emission_probs(state)
            state_trans_probs = hmm.get_transition_probs(state)

            # If we are looking at the first observation
            if i == 0:
                # Calculate the initial probability o per state
                state_prob = state_init_probs* state_emit_probs[observation]
                # Use natural log to prevent numerical underflow
                prob_matrix[j][i] = round(np.log(state_prob), 2)
            # Else if we are looking at the second observation onward
            else:
                # Calculate the possible probabilities based on transitioning for all states
                possible_probs = [np.exp(prob_matrix[k][i-1]) * state_trans_probs[prev_state] * state_emit_probs[observation] for k, prev_state in enumerate(states)]
                # Get the max probability and add the log value to the viterbi matrix
                max_prob = max(possible_probs)
                prob_matrix[j][i] = round(np.log(max_prob), 2)
                # Get the index of the maximum probabilities and add that to the traceback matrix
                previous_coords = np.argmax(possible_probs)
                traceback_matrix[j][i] = previous_coords

    return prob_matrix, traceback_matrix



In [286]:
def viterbi_traceback(viterbi_matrix, traceback_matrix):

    # Identify the final state with the highest probability
    end_state = np.argmax(viterbi_matrix[:,-1])

    # Add state to list
    predictions = [int(end_state)]

    # Traceback through the matrix starting at the end of the matrix
    for i in range(len(obs)-1, 0, -1):
        # Append the state index to the predictions
        end_state = traceback_matrix[end_state][i]
        predictions.append(int(end_state))

    # Reverse the predictions so it starts at the beginning of matrix
    return predictions[::-1]

In [287]:
 def convert_index_to_states(predictions, states):
    states_final = []
    for index in predictions:
        states_final.append(states[index])
    return states_final

In [288]:

# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}
path1 = viterbi_algorithm(obs, init_probs, trans_probs, emit_probs)


['I', 'G']
Observation: ACGCGATC
Viterbi matrix:
[[ -4.61  -2.85  -4.28  -5.71  -7.14  -9.95 -12.76 -13.21]
 [ -1.02  -3.43  -5.84  -8.25 -10.32 -10.36 -11.38 -13.79]]
Traceback matrix:
[[0 1 0 0 0 0 0 1]
 [0 1 1 1 0 0 1 1]]
Optimal path index:
[1, 0, 0, 0, 0, 1, 1, 0]
Optimal path:
['G', 'I', 'I', 'I', 'I', 'G', 'G', 'I']



In [290]:
# Example observation sequence
observations = ["GGCACTGAA", "ACGCGATC"]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "CpG": 0.3,
    "Genome": 0.5,
    "Promoter": 0.2
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "CpG": {"CpG": 0.6, "Promoter": 0.2, "Genome": 0.2},
    "Genome": {"CpG": 0.2, "Promoter": 0.1, "Genome": 0.7},
    "Promoter": {"CpG": 0.1, "Promoter": 0.7, "Genome": 0.2},
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "CpG": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "Genome": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "Promoter": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
}
path2 = viterbi_algorithm(observations, init_probs, trans_probs, emit_probs)

['CpG', 'Genome', 'Promoter']
Observation: GGCACTGAA
Viterbi matrix:
[[ -2.12  -3.55  -4.98  -7.79  -9.22 -12.03 -13.46 -16.27 -18.76]
 [ -2.3   -4.27  -6.24  -7.79  -9.76 -11.32 -13.29 -14.85 -16.41]
 [ -2.81  -4.37  -5.93  -7.9   -9.46 -11.43 -12.99 -14.96 -16.93]]
Traceback matrix:
[[0 0 0 0 0 0 0 0 1]
 [0 1 1 0 1 1 1 1 1]
 [0 2 2 2 2 2 2 2 2]]
Optimal path index:
[0, 0, 0, 1, 1, 1, 1, 1]
Optimal path:
['CpG', 'CpG', 'CpG', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome']

Observation: ACGCGATC
Viterbi matrix:
[[ -3.51  -4.43  -5.86  -7.29  -8.72 -11.53 -14.34 -15.43]
 [ -1.9   -3.87  -5.84  -7.81  -9.78 -11.34 -12.9  -14.87]
 [ -3.22  -4.71  -6.27  -7.83  -9.39 -11.36 -13.33 -14.89]]
Traceback matrix:
[[0 1 0 0 0 0 0 1]
 [0 1 1 1 1 1 1 1]
 [0 1 2 2 2 2 2 2]]
Optimal path index:
[1, 1, 1, 1, 1, 1, 1, 1]
Optimal path:
['Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome']



In [258]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}
path3 = viterbi_algorithm(obs, init_probs, trans_probs, emit_probs)

Observation: GGCACTGAA
Viterbi matrix:
[[ -2.53  -3.8   -5.07  -7.73  -8.68 -11.34 -11.7  -14.36 -16.11]
 [ -1.83  -3.54  -5.25  -6.56  -8.27  -9.58 -11.29 -12.6  -13.91]]
Traceback matrix:
[[0 0 0 0 1 0 1 0 1]
 [0 1 1 1 1 1 1 1 1]]
Optimal path:
['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']

